In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import requests
import os
import pandas as pd
import numpy as np
import sys
from pathlib import Path
sys.path.append(os.path.abspath(".."))
import src.raw_preprocessing as rp
import src.feature_engineering as fe
import src.create_dataset as cd
from datetime import date, datetime, timedelta
from vacances_scolaires_france import SchoolHolidayDates

import openmeteo_requests
import requests_cache
from retry_requests import retry
from joblib import load

In [134]:
conso = pd.read_parquet("../data/final_datasets/datasets_linear_models/conso_v3_linear.parquet")

In [135]:
conso.columns

Index(['Consommation', 'Zone_A', 'Zone_B', 'Zone_C',
       'Vacances de la Toussaint', 'Vacances de Noël', 'Vacances d'Hiver',
       'Vacances de Printemps', 'Vacances d'Été', 'public_holidays', '44T',
       '69T', '59T', '75T', '13T', '33T', 'T', 'U', 'FF', 'PMER', 'RR1',
       'year', 'month', 'hour', 'day_of_week', 'is_weekend', 'hour_sin',
       'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'month_sin',
       'month_cos', 'lagged_1', 'lagged_2', 'lagged_48', 'lagged_336',
       'rolling_mean_24h', 'rolling_std_24h', 'rolling_mean_7d',
       'rolling_std_7d', 'rolling_max_24h', 'rolling_min_24h',
       'consumption_diff_1', 'consumption_diff_48', 'consumption_pct_change_1',
       'consumption_pct_change_48', 'season_Spring', 'season_Summer',
       'season_Winter', 'temp_sq', 'humidity_sq', 'hour_x_is_weekend',
       'hour_x_is_holiday', 'hour_x_dow', 'hour_x_month', 'is_weekend_x_month',
       'is_holiday_x_month', 'hour_x_temp', 'hour_x_humidity', 'hour_x_wind',
  

In [3]:
cd.download_monthly_data()

the download of conso_energie_2026.zip has started


In [69]:
df = rp.conso_preprocess(Path("../data/conso/real_time_conso/"))

df = df[df["Heures"].apply(lambda x: x.minute in {00, 30})]

df = df.reset_index(drop=True)
df.loc[len(df)] = None

In [70]:
df.columns

Index(['Date', 'Heures', 'Consommation'], dtype='object')

In [71]:
df

,Date,Heures,Consommation
0,2026-05-01,00:00:00,41688
1,2026-05-01,00:30:00,40393
2,2026-05-01,01:00:00,38108
3,2026-05-01,01:30:00,37744
4,2026-05-01,02:00:00,36708
...,...,...,...
4939,2026-08-11,21:30:00,45757
4940,2026-08-11,22:00:00,45069
4941,2026-08-11,22:30:00,45359
4942,2026-08-11,23:00:00,45241


In [72]:
df = fe.lagged_consumption(df)

In [73]:
t = df["Heures"].iloc[-2]

new_time = (
    datetime.combine(datetime.today(), t)
    + timedelta(minutes=30)
).time()

In [74]:
df.loc[len(df)-1, "Heures"] = new_time

In [75]:
df.loc[len(df)-1, "Date"] = datetime.today().strftime('%Y-%m-%d')

In [76]:
df

,Date,Heures,Consommation,lagged_1,lagged_2,lagged_48,lagged_336
0,2026-05-01,00:00:00,41688,None,None,None,None
1,2026-05-01,00:30:00,40393,41688,None,None,None
2,2026-05-01,01:00:00,38108,40393,41688,None,None
3,2026-05-01,01:30:00,37744,38108,40393,None,None
4,2026-05-01,02:00:00,36708,37744,38108,None,None
...,...,...,...,...,...,...,...
4939,2026-08-11,21:30:00,45757,45891,46801,45727,46056
4940,2026-08-11,22:00:00,45069,45757,45891,45151,45724
4941,2026-08-11,22:30:00,45359,45069,45757,44957,46057
4942,2026-08-11,23:00:00,45241,45359,45069,44489,46060


#### Adding the name of the holidays

In [77]:
from pprint import pprint
today = date(2026, 5, 16).isoformat()

url = "https://data.education.gouv.fr/api/explore/v2.1/catalog/datasets/fr-en-calendrier-scolaire/records"

params = {
    "where": f"start_date <= date'{today}' AND end_date >= date'{today}' AND zones = 'Zone C'"
}

response = requests.get(url, params=params)

In [78]:
#Adding the infos about the holidays
today = date.today().isoformat()
url = "https://data.education.gouv.fr/api/explore/v2.1/catalog/datasets/fr-en-calendrier-scolaire/records"

zones = ['A', 'B', 'C']
vacances = ['vacances de la toussaint', 'vacances de noël', "vacances d'hiver",'vacances de printemps', "vacances d'été"]
feries = [
    "jour de l'an",
    "lundi de pâques",
    "fête du travail",
    "victoire 1945",
    "ascension",
    "lundi de pentecôte",
    "fête nationale",
    "assomption",
    "toussaint",
    "armistice",
    "noël",
    "pont de l'ascension",
]

#finished_with_holidays = False
df.loc[:, "Zone_A"] = 0
df.loc[:, "Zone_B"] = 0
df.loc[:, "Zone_C"] = 0

df.loc[:, "public_holidays"] = 0
df.loc[:, "Vacances de la Toussaint"] = 0
df.loc[:, "Vacances de Noël"] = 0
df.loc[:, "Vacances d'Hiver"] = 0
df.loc[:, "Vacances de Printemps"] = 0
df.loc[:, "Vacances d'Été"] = 0

for z in zones:
    params = {
        "where": f"start_date <= date'{today}' AND end_date >= date'{today}' AND zones = 'Zone {z}'"
    }
    response = requests.get(url, params=params).json()
    # If no holidays we set the columns with the value 0
    if response["total_count"] == 0:
        continue

    else : 
        
        for event in response["results"]:
            # We check if it's school holidays
            if event["description"].lower() in vacances:
                df.loc[:, f"Zone_{z}"] = 1
                if event["description"].lower() == "vacances de la toussaint":
                    df.loc[:, "Vacances de la Toussaint"] = 1
                    
                elif event["description"].lower() == "vacances de noël":
                    df.loc[:, "Vacances de Noël"] = 1
                    
                elif event["description"].lower() == "vacances d'hiver":
                    df.loc[:, "Vacances d'Hiver"] = 1
                    
                elif event["description"].lower() == "vacances de printemps":
                    df.loc[:, "Vacances de Printemps"] = 1

                elif event["description"].lower() == "vacances d'été":
                    df.loc[:, "Vacances d'Été"] = 1
            # Or if it's public holidays (jours fériés)
            elif event in feries:
                df.loc[:, "public_holidays"] = 1

In [79]:
df["Consommation"] = pd.to_numeric(df["Consommation"], errors="coerce")
df["Consommation"] = df["Consommation"].interpolate()

In [80]:
df = fe.date_and_hour_pred(df)
df = fe.cyclical_encoding(df)
df = fe.rolling_window(df)
df = fe.lagged_trend(df)
df = fe.seasons_linear(df)
df = df.drop(["Consommation"], axis=1)

In [81]:
df.columns

Index(['lagged_1', 'lagged_2', 'lagged_48', 'lagged_336', 'Zone_A', 'Zone_B',
       'Zone_C', 'public_holidays', 'Vacances de la Toussaint',
       'Vacances de Noël', 'Vacances d'Hiver', 'Vacances de Printemps',
       'Vacances d'Été', 'full_date', 'year', 'month', 'hour', 'day_of_week',
       'is_weekend', 'hour_sin', 'hour_cos', 'day_of_week_sin',
       'day_of_week_cos', 'month_sin', 'month_cos', 'rolling_mean_24h',
       'rolling_std_24h', 'rolling_mean_7d', 'rolling_std_7d',
       'rolling_max_24h', 'rolling_min_24h', 'consumption_diff_1',
       'consumption_diff_48', 'consumption_pct_change_1',
       'consumption_pct_change_48', 'season_Spring', 'season_Summer',
       'season_Winter'],
      dtype='object')

In [108]:
row = df.iloc[-1]

pred = pd.DataFrame([row] * 10)

pred["full_date"] = row["full_date"] + pd.to_timedelta(range(10), unit="m") * 30
pred = pred.reset_index(drop=True)

pred.columns

Index(['lagged_1', 'lagged_2', 'lagged_48', 'lagged_336', 'Zone_A', 'Zone_B',
       'Zone_C', 'public_holidays', 'Vacances de la Toussaint',
       'Vacances de Noël', 'Vacances d'Hiver', 'Vacances de Printemps',
       'Vacances d'Été', 'full_date', 'year', 'month', 'hour', 'day_of_week',
       'is_weekend', 'hour_sin', 'hour_cos', 'day_of_week_sin',
       'day_of_week_cos', 'month_sin', 'month_cos', 'rolling_mean_24h',
       'rolling_std_24h', 'rolling_mean_7d', 'rolling_std_7d',
       'rolling_max_24h', 'rolling_min_24h', 'consumption_diff_1',
       'consumption_diff_48', 'consumption_pct_change_1',
       'consumption_pct_change_48', 'season_Spring', 'season_Summer',
       'season_Winter'],
      dtype='object')

In [111]:
pred["full_date"].iloc[0]

Timestamp('2026-08-11 23:30:00')

### RTE France API

In [86]:
id_client = "863fa354-33b4-4bf6-a844-3dd668062f92"
id_secret = "83b2284d-b40f-4627-916f-d35473030cbf"
url = "https://digital.iservices.rte-france.com/token/oauth/"

response = requests.post(url, auth=(id_client, id_secret))

In [87]:
response.json()

{'access_token': 'k474YPtYGg8Im6fzkvcoIUeWaPI4fzxRjj6ankSZK5EghTijE48cHG',
 'token_type': 'Bearer',
 'expires_in': 3600}

In [88]:
token = response.json()["access_token"]
headers = {
    "Authorization" : f"Bearer {token}"
}

url = "https://digital.iservices.rte-france.com/open_api/consumption/v1/short_term"

data = requests.get(url, headers=headers)

In [89]:
data.json()

{'short_term': [{'type': 'REALISED',
   'start_date': '2026-07-17T00:00:00+02:00',
   'end_date': '2026-07-18T00:00:00+02:00',
   'values': [{'start_date': '2026-07-17T00:00:00+02:00',
     'end_date': '2026-07-17T00:15:00+02:00',
     'updated_date': '2026-07-17T13:05:47+02:00',
     'value': 47387},
    {'start_date': '2026-07-17T00:15:00+02:00',
     'end_date': '2026-07-17T00:30:00+02:00',
     'updated_date': '2026-07-17T13:05:47+02:00',
     'value': 46988},
    {'start_date': '2026-07-17T00:30:00+02:00',
     'end_date': '2026-07-17T00:45:00+02:00',
     'updated_date': '2026-07-17T13:05:48+02:00',
     'value': 45765},
    {'start_date': '2026-07-17T00:45:00+02:00',
     'end_date': '2026-07-17T01:00:00+02:00',
     'updated_date': '2026-07-17T13:05:48+02:00',
     'value': 44762},
    {'start_date': '2026-07-17T01:00:00+02:00',
     'end_date': '2026-07-17T01:15:00+02:00',
     'updated_date': '2026-07-17T13:05:49+02:00',
     'value': 43709},
    {'start_date': '2026-07-17T01

### Open-Meteo API

In [112]:
# ---- Open-Meteo -----
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
lat_long = {
    '13': {'ville': 'Marseille', 'latitude': 43.2965, 'longitude': 5.3698},     # Bouches-du-Rhône
    '33': {'ville': 'Bordeaux',  'latitude': 44.8378, 'longitude': -0.5792},    # Gironde
    '44': {'ville': 'Nantes',    'latitude': 47.2184, 'longitude': -1.5536},    # Loire-Atlantique
    '59': {'ville': 'Lille',     'latitude': 50.6292, 'longitude': 3.0573},     # Nord
    '69': {'ville': 'Lyon',      'latitude': 45.7640, 'longitude': 4.8357},     # Rhône
    '75': {'ville': 'Paris',     'latitude': 48.8566, 'longitude': 2.3522},     # Paris
}

station_population = {
    '13': 2087658,   # Bouches-du-Rhône 
    '33': 1690493,   # Gironde           
    '44': 1487570,   # Loire-Atlantique   
    '59': 2615635,   # Nord              
    '69': 1914667,   # Rhône             
    '75': 2103778,   # Paris
}

total_pop = sum(station_population.values())
weights = {city: pop / total_pop for city, pop in station_population.items()}


cols = ['T', 'U', 'FF', 'PMER', 'RR1']
df_temp = pd.DataFrame(
    np.zeros(shape=(pred.shape[0], len(cols))),
    columns=cols
)

for k, v in lat_long.items():

    # Calling the API for our department
    params = {
        "latitude": v["latitude"],
        "longitude": v["longitude"],
        "hourly": ["temperature_2m", "relative_humidity_2m", "rain", "surface_pressure", "wind_speed_10m"],
        "timezone": "Europe/Paris",
        "past_days": 7,
        "forecast_days": 2,
    }
    responses = openmeteo.weather_api(url, params = params)
    
    # Process first location. Add a for-loop for multiple locations or weather models
    response = responses[0]

    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
    hourly_rain = hourly.Variables(2).ValuesAsNumpy()
    hourly_surface_pressure = hourly.Variables(3).ValuesAsNumpy()
    hourly_wind_speed_10m = hourly.Variables(4).ValuesAsNumpy()
    
    hourly_data = {
        "date": pd.date_range(
            start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
            end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
            freq = pd.Timedelta(seconds = hourly.Interval()),
            inclusive = "left"
        ).tz_convert(response.Timezone().decode())
    }

    # Preprocessing the weather date
    hourly_data["temperature_2m"] = hourly_temperature_2m
    hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
    hourly_data["rain"] = hourly_rain
    hourly_data["surface_pressure"] = hourly_surface_pressure
    hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
    
    hourly_dataframe = pd.DataFrame(data = hourly_data)

    hourly_dataframe = hourly_dataframe.rename(columns={
        "temperature_2m" : "T",
        "relative_humidity_2m": "U",
        "rain" : "RR1",
        "surface_pressure" : "PMER",
        "wind_speed_10m" : "FF"
    })

    # We add the 30min weather date 
    hourly_dataframe["date"] = pd.to_datetime(hourly_dataframe["date"])
    hourly_dataframe = hourly_dataframe.set_index("date")
    hourly_dataframe = hourly_dataframe.asfreq("30min")

    # We interpolate the missing values
    hourly_dataframe["RR1"] = hourly_dataframe["RR1"].fillna(0)
    hourly_dataframe = hourly_dataframe.interpolate(method="time")
    hourly_dataframe = hourly_dataframe.reset_index()
    hourly_dataframe["date"] = hourly_dataframe["date"].dt.tz_localize(None)
    hourly_dataframe = hourly_dataframe[hourly_dataframe["date"].isin(pred["full_date"])]
    hourly_dataframe = hourly_dataframe.reset_index()
    
    pred[f"{k}T"] = hourly_dataframe[['T']]


    population = weights[k]
    df_temp += hourly_dataframe[cols].values * population


pred = pd.concat([pred, df_temp], axis=1)
pred = pred.drop("full_date", axis=1)

In [113]:
pred

,lagged_1,lagged_2,lagged_48,lagged_336,Zone_A,Zone_B,Zone_C,public_holidays,Vacances de la Toussaint,Vacances de Noël,...,33T,44T,59T,69T,75T,T,U,FF,PMER,RR1
0,45241,45359,43363,44821,1,1,1,0,0,0,...,27.445499,30.786999,21.505499,28.160999,26.639999,26.801603,40.959138,8.469827,1013.968758,0.0
1,45241,45359,43363,44821,1,1,1,0,0,0,...,26.795500,30.087000,20.980499,27.861000,25.915001,26.220270,42.393697,7.935935,1014.074799,0.0
2,45241,45359,43363,44821,1,1,1,0,0,0,...,26.345501,30.062000,20.505499,27.636002,25.264999,25.818747,43.646723,8.253386,1014.160698,0.0
3,45241,45359,43363,44821,1,1,1,0,0,0,...,25.895500,30.036999,20.030499,27.411001,24.615000,25.417224,44.899750,8.570838,1014.246559,0.0
4,45241,45359,43363,44821,1,1,1,0,0,0,...,25.595501,29.312000,19.630499,27.086000,24.065001,25.015825,46.117971,7.770242,1014.253197,0.0
5,45241,45359,43363,44821,1,1,1,0,0,0,...,25.295500,28.587000,19.230499,26.761000,23.515001,24.614426,47.336195,6.969646,1014.259880,0.0
6,45241,45359,43363,44821,1,1,1,0,0,0,...,25.095501,28.130751,18.880499,26.410999,23.090000,24.315113,47.986769,7.427150,1014.256065,0.0
7,45241,45359,43363,44821,1,1,1,0,0,0,...,24.895500,27.674501,18.530499,26.061001,22.665001,24.015802,48.637342,7.884653,1014.252213,0.0
8,45241,45359,43363,44821,1,1,1,0,0,0,...,24.670500,27.393250,18.180500,25.636002,22.264999,23.711816,49.344408,7.255541,1014.305885,0.0
9,45241,45359,43363,44821,1,1,1,0,0,0,...,24.445499,27.112000,17.830500,25.211000,21.865000,23.407830,50.051475,6.626429,1014.359550,0.0


In [114]:
print(pred.columns)
pred

Index(['lagged_1', 'lagged_2', 'lagged_48', 'lagged_336', 'Zone_A', 'Zone_B',
       'Zone_C', 'public_holidays', 'Vacances de la Toussaint',
       'Vacances de Noël', 'Vacances d'Hiver', 'Vacances de Printemps',
       'Vacances d'Été', 'year', 'month', 'hour', 'day_of_week', 'is_weekend',
       'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos',
       'month_sin', 'month_cos', 'rolling_mean_24h', 'rolling_std_24h',
       'rolling_mean_7d', 'rolling_std_7d', 'rolling_max_24h',
       'rolling_min_24h', 'consumption_diff_1', 'consumption_diff_48',
       'consumption_pct_change_1', 'consumption_pct_change_48',
       'season_Spring', 'season_Summer', 'season_Winter', '13T', '33T', '44T',
       '59T', '69T', '75T', 'T', 'U', 'FF', 'PMER', 'RR1'],
      dtype='object')


,lagged_1,lagged_2,lagged_48,lagged_336,Zone_A,Zone_B,Zone_C,public_holidays,Vacances de la Toussaint,Vacances de Noël,...,33T,44T,59T,69T,75T,T,U,FF,PMER,RR1
0,45241,45359,43363,44821,1,1,1,0,0,0,...,27.445499,30.786999,21.505499,28.160999,26.639999,26.801603,40.959138,8.469827,1013.968758,0.0
1,45241,45359,43363,44821,1,1,1,0,0,0,...,26.795500,30.087000,20.980499,27.861000,25.915001,26.220270,42.393697,7.935935,1014.074799,0.0
2,45241,45359,43363,44821,1,1,1,0,0,0,...,26.345501,30.062000,20.505499,27.636002,25.264999,25.818747,43.646723,8.253386,1014.160698,0.0
3,45241,45359,43363,44821,1,1,1,0,0,0,...,25.895500,30.036999,20.030499,27.411001,24.615000,25.417224,44.899750,8.570838,1014.246559,0.0
4,45241,45359,43363,44821,1,1,1,0,0,0,...,25.595501,29.312000,19.630499,27.086000,24.065001,25.015825,46.117971,7.770242,1014.253197,0.0
5,45241,45359,43363,44821,1,1,1,0,0,0,...,25.295500,28.587000,19.230499,26.761000,23.515001,24.614426,47.336195,6.969646,1014.259880,0.0
6,45241,45359,43363,44821,1,1,1,0,0,0,...,25.095501,28.130751,18.880499,26.410999,23.090000,24.315113,47.986769,7.427150,1014.256065,0.0
7,45241,45359,43363,44821,1,1,1,0,0,0,...,24.895500,27.674501,18.530499,26.061001,22.665001,24.015802,48.637342,7.884653,1014.252213,0.0
8,45241,45359,43363,44821,1,1,1,0,0,0,...,24.670500,27.393250,18.180500,25.636002,22.264999,23.711816,49.344408,7.255541,1014.305885,0.0
9,45241,45359,43363,44821,1,1,1,0,0,0,...,24.445499,27.112000,17.830500,25.211000,21.865000,23.407830,50.051475,6.626429,1014.359550,0.0


In [115]:
pred = fe.interactions_linear(pred)

In [116]:
print(len(pred.columns))
pred

82


,lagged_1,lagged_2,lagged_48,lagged_336,Zone_A,Zone_B,Zone_C,public_holidays,Vacances de la Toussaint,Vacances de Noël,...,is_weekend_x_season_Summer,is_holiday_x_season_Summer,hour_x_season_Winter,is_weekend_x_season_Winter,is_holiday_x_season_Winter,temp_x_humidity,temp_x_wind,humidity_x_wind,HDD,CDD
0,45241,45359,43363,44821,1,1,1,0,0,0,...,0,0,0.0,0,0,1097.770557,227.004943,346.916815,0.0,8.801603
1,45241,45359,43363,44821,1,1,1,0,0,0,...,0,0,0.0,0,0,1111.574183,208.082354,336.433613,0.0,8.220270
2,45241,45359,43363,44821,1,1,1,0,0,0,...,0,0,0.0,0,0,1126.903707,213.092095,360.233271,0.0,7.818747
3,45241,45359,43363,44821,1,1,1,0,0,0,...,0,0,0.0,0,0,1141.227009,217.846915,384.828484,0.0,7.417224
4,45241,45359,43363,44821,1,1,1,0,0,0,...,0,0,0.0,0,0,1153.679092,194.379020,358.347805,0.0,7.015825
5,45241,45359,43363,44821,1,1,1,0,0,0,...,0,0,0.0,0,0,1165.153253,171.553844,329.916542,0.0,6.614426
6,45241,45359,43363,44821,1,1,1,0,0,0,...,0,0,0.0,0,0,1166.803707,180.591981,356.404908,0.0,6.315113
7,45241,45359,43363,44821,1,1,1,0,0,0,...,0,0,0.0,0,0,1168.064782,189.356269,383.488569,0.0,6.015802
8,45241,45359,43363,44821,1,1,1,0,0,0,...,0,0,0.0,0,0,1170.045528,172.042049,358.020365,0.0,5.711816
9,45241,45359,43363,44821,1,1,1,0,0,0,...,0,0,0.0,0,0,1171.596443,155.110320,331.662530,0.0,5.407830


In [117]:
pred.columns

Index(['lagged_1', 'lagged_2', 'lagged_48', 'lagged_336', 'Zone_A', 'Zone_B',
       'Zone_C', 'public_holidays', 'Vacances de la Toussaint',
       'Vacances de Noël', 'Vacances d'Hiver', 'Vacances de Printemps',
       'Vacances d'Été', 'year', 'month', 'hour', 'day_of_week', 'is_weekend',
       'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos',
       'month_sin', 'month_cos', 'rolling_mean_24h', 'rolling_std_24h',
       'rolling_mean_7d', 'rolling_std_7d', 'rolling_max_24h',
       'rolling_min_24h', 'consumption_diff_1', 'consumption_diff_48',
       'consumption_pct_change_1', 'consumption_pct_change_48',
       'season_Spring', 'season_Summer', 'season_Winter', '13T', '33T', '44T',
       '59T', '69T', '75T', 'T', 'U', 'FF', 'PMER', 'RR1', 'temp_sq',
       'humidity_sq', 'hour_x_is_weekend', 'hour_x_is_holiday', 'hour_x_dow',
       'hour_x_month', 'is_weekend_x_month', 'is_holiday_x_month',
       'hour_x_temp', 'hour_x_humidity', 'hour_x_wind', 'is_weekend_x_temp

In [118]:
models = load("../artifacts/model_artifacts/Ridge_2026-07-30_23-24-53/Ridge_models.joblib")

In [119]:
pred2 = pred.copy()
pred2 = fe.drop_useless(pred2)
pred2.columns

Index(['lagged_1', 'lagged_2', 'lagged_48', 'lagged_336', 'Zone_A', 'Zone_B',
       'Zone_C', 'public_holidays', 'Vacances de la Toussaint',
       'Vacances de Noël', 'Vacances d'Hiver', 'Vacances de Printemps',
       'Vacances d'Été', 'year', 'is_weekend', 'hour_sin', 'hour_cos',
       'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos',
       'rolling_mean_24h', 'rolling_std_24h', 'rolling_mean_7d',
       'rolling_std_7d', 'rolling_max_24h', 'rolling_min_24h',
       'consumption_diff_1', 'consumption_diff_48', 'consumption_pct_change_1',
       'consumption_pct_change_48', 'season_Spring', 'season_Summer',
       'season_Winter', '13T', '33T', '44T', '59T', '69T', '75T', 'T', 'U',
       'FF', 'PMER', 'RR1', 'temp_sq', 'humidity_sq', 'hour_x_is_weekend',
       'hour_x_is_holiday', 'hour_x_dow', 'hour_x_month', 'is_weekend_x_month',
       'is_holiday_x_month', 'hour_x_temp', 'hour_x_humidity', 'hour_x_wind',
       'is_weekend_x_temp', 'is_holiday_x_temp', 'month_x

In [120]:
predictions = {}
for i in range(0, 10):
    if i == 0:
        p = models[f"Ridge_{i}"].predict(pred)
        predictions[f"horizon_{i}"] = p[0]
    else:
        p = models[f"Ridge_{i}"].predict(pred2.iloc[i-1:i])
        predictions[f"horizon_{i}"] = p[0]

In [121]:
predictions

{'horizon_0': np.float64(43920.25638597465),
 'horizon_1': np.float64(43013.350368842366),
 'horizon_2': np.float64(42642.43310292787),
 'horizon_3': np.float64(41544.289861585195),
 'horizon_4': np.float64(40024.864300719964),
 'horizon_5': np.float64(39904.91150065894),
 'horizon_6': np.float64(39852.27695117158),
 'horizon_7': np.float64(39802.39570840005),
 'horizon_8': np.float64(39860.16820857554),
 'horizon_9': np.float64(40981.969675001965)}

In [197]:
print(type(pred2.iloc[9]))
print(pred2.iloc[9].shape)

print(type(pred2.iloc[[9]]))
print(pred2.iloc[[9]].shape)

<class 'pandas.core.series.Series'>
(79,)
<class 'pandas.core.frame.DataFrame'>
(1, 79)
